In [1]:
import pandas as pd
import numpy as np
import graphviz
import json
import os
from dep1 import DataTransformer
from dep2 import safe_write_json


In [ ]:
with open('data1.json', 'r') as f:
    data1 = json.load(f)

with open('data2.json', 'r') as f:
    data2 = json.load(f)

print(f"Loaded {len(data1)} records from data1.json and {len(data2)} records from data2.json")


In [ ]:
transformer = DataTransformer()

stats1 = transformer.get_session_stats(data1, 'd1')
page_views1 = transformer.get_page_views(data1, 'd1')

stats2 = transformer.get_session_stats(data2, 'd2')
page_views2 = transformer.get_page_views(data2, 'd2')

print("Session Stats 1:\\n", stats1.head())
print("\\nPage Views 1:\\n", page_views1.head())
print("\\nSession Stats 2:\\n", stats2.head())
print("\\nPage Views 2:\\n", page_views2.head())


In [ ]:
# Merge session stats
merged_stats = pd.merge(stats1, stats2, on='user_id', how='outer').fillna(0)
merged_stats['total_duration'] = merged_stats['d1_duration_sum'] + merged_stats['d2_duration_sum']
merged_stats['total_sessions'] = merged_stats['d1_session_count'] + merged_stats['d2_session_count']

# Merge page views
merged_page_views = pd.merge(page_views1, page_views2, on='page', how='outer').fillna(0)
merged_page_views['total_views'] = merged_page_views['d1_view_count'] + merged_page_views['d2_view_count']

print("\\nMerged Session Stats:\\n", merged_stats)
print("\\nMerged Page Views:\\n", merged_page_views)

# Write results
stats_write_message = safe_write_json(merged_stats.to_dict(orient='records'), 'session_stats.json', folder='outputdirectory')
views_write_message = safe_write_json(merged_page_views.to_dict(orient='records'), 'page_views.json', folder='outputdirectory')

print(stats_write_message)
print(views_write_message)


In [ ]:
dot = graphviz.Digraph(comment='ETL Pipeline')
dot.node('d1', 'data1.json')
dot.node('d2', 'data2.json')
dot.node('trans', 'DataTransformer')
dot.node('stats', 'Session Stats')
dot.node('views', 'Page Views')
dot.node('merge', 'Merge & Aggregate')
dot.node('out_stats', 'session_stats.json')
dot.node('out_views', 'page_views.json')

dot.edge('d1', 'trans')
dot.edge('d2', 'trans')
dot.edge('trans', 'stats')
dot.edge('trans', 'views')
dot.edge('stats', 'merge')
dot.edge('views', 'merge')
dot.edge('merge', 'out_stats')
dot.edge('merge', 'out_views')

# Ensure output directory exists for graphviz
if not os.path.exists('outputdirectory'):
    os.makedirs('outputdirectory')

# Save graph
dot.render('outputdirectory/etl_pipeline', format='png', cleanup=True)
print("ETL pipeline graph saved to outputdirectory/etl_pipeline.png")
print("Contents of outputdirectory:", os.listdir('outputdirectory'))
